### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [1]:
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [2]:
loader=TextLoader("langchain_crewai_dataset.txt")
docs=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks=text_splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

vectorstore

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriver =vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5})
vectorstore,retriver

(<langchain_community.vectorstores.faiss.FAISS at 0x26792998ad0>,
 VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026792998AD0>, search_type='mmr', search_kwargs={'k': 5}))

LLM and Prompt

In [8]:
from langchain_ollama import ChatOllama , OllamaEmbeddings
llm=ChatOllama(model="llama3.2:latest") 
llm

ChatOllama(model='llama3.2:latest')

In [9]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOllama(model='llama3.2:latest')
| StrOutputParser()

In [10]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'Here\'s an expanded query that incorporates relevant synonyms, technical terms, and useful context:\n\n"Langchain memory, language model memory, transformer-based memory, large language models, conversational AI, natural language processing (NLP), deep learning, neural networks."\n\nAlternatively, you can also use phrases like:\n\n* "Memory module in Langchain architecture"\n* "Token-based memory for NLP applications"\n* "Large-scale language understanding using distributed memory"\n* "Transformers with memory capabilities for NLP tasks"\n\nThese expanded queries cover different aspects of the original query and provide more context to improve document retrieval.'

In [11]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [17]:
rag_pipeline=(
    RunnableMap({
        "input": lambda x:x['input'],
        "context":lambda x:retriver.invoke(query_expansion_chain.invoke({"query":x['input']}))}
    )|document_chain
)

In [18]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Here's an expanded version of the query:

{"
  'input': 'types of memory supported by LangChain',
  'related concepts': ['memory types', 'LangChain capabilities'],
  'technical terms': ['ephemeral storage', 'persistent storage', 'temporal storage'],
  'useful context': {
    'domain knowledge': 'natural language processing, artificial intelligence',
    'application scenarios': ['conversational AI, chatbots']
  }
}"
✅ Answer:
 According to the context, LangChain supports two types of memory modules:

1. ConversationBufferMemory
2. ConversationSummaryMemory

These memory modules allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits.


In [19]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

To expand the original query and improve document retrieval, we can add relevant synonyms, technical terms, and useful context. Here's an example of an expanded query:

**Expanded Query:** 
```
{
  "input": "CrewAI agent OR crewed AI OR autonomous agent",
  "context": {
    "domain": "Artificial Intelligence",
    "topical area": "Machine Learning"
  },
  "synonyms": ["agent", "bot", "automaton"],
  "technical terms": ["CrewAI", "autonomous system", "machine learning model"]
}
```
In this expanded query, we've added:

1. **Synonyms**: We've included relevant synonyms for the original search term "CrewAI agent" to capture variations in language and ensure more accurate results. The added synonyms are:
	* "agent"
	* "bot"
	* "automaton"
2. **Context**: We've provided context about the domain (Artificial Intelligence) and topical area (Machine Learning) where the query is most relevant. This helps narrow down search results to only those that are directly related to these topics.
3. **Tec